In [1]:
# ------------------------------------------------------------------------------
# SEL 1: IMPORT LIBRARIES
# ------------------------------------------------------------------------------
import datetime
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

print("✅ Pustaka berhasil dimuat.")

✅ Pustaka berhasil dimuat.


In [6]:
# ------------------------------------------------------------------------------
# SEL 2: LOAD DATASET & CLEANING
# ------------------------------------------------------------------------------

import pandas as pd
from pathlib import Path

# 1. Cari file Excel di folder BAB IV dan seluruh subfoldernya
folder_bab4 = Path.cwd().parent
nama_file = "data greenline 2024 sd juni 2026.xlsx"

hasil_pencarian = list(folder_bab4.rglob(nama_file))

if not hasil_pencarian:
    raise FileNotFoundError(
        f"File '{nama_file}' tidak ditemukan di:\n"
        f"{folder_bab4.resolve()}\n\n"
        "Periksa kembali nama file dan lokasi folder DATASET."
    )

file_path = hasil_pencarian[0]

print(f"File ditemukan: {file_path.resolve()}")

# Load file Excel
df_raw = pd.read_excel(file_path)

# 2. Bersihkan kolom jumlah penumpang
kolom_penumpang = [
    "penumpang_berangkat_komuter",
    "penumpang_datang_komuter",
]

for kolom in kolom_penumpang:
    df_raw[kolom] = pd.to_numeric(
        df_raw[kolom]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="raise",
    ).astype(int)

# 3. Format tanggal ke datetime
df_raw["tanggal"] = pd.to_datetime(
    df_raw["tanggal"],
    format="%d/%m/%Y",
    errors="raise",
)

# 4. Buat variabel target
df_raw["volume_penumpang"] = (
    df_raw["penumpang_berangkat_komuter"]
    + df_raw["penumpang_datang_komuter"]
)

# 5. Urutkan berdasarkan stasiun dan tanggal
df_sorted = (
    df_raw
    .sort_values(by=["nama_stasiun", "tanggal"])
    .reset_index(drop=True)
)

# 6. Tampilkan hasil pemeriksaan
print("\n=== OUTPUT UNTUK TULISAN DI WORD (SUB-BAB 4.1) ===")
print(f"Lokasi file                      : {file_path.resolve()}")
print(f"Total Baris Data Mentah (N_raw) : {len(df_sorted)} baris")
print(
    f"Jumlah Sel Kosong (Null Check)   : "
    f"{df_sorted.isnull().sum().sum()} sel"
)

File ditemukan: C:\Users\HP\Documents\Tito\BAB IV\DATASET\data greenline 2024 sd juni 2026.xlsx

=== OUTPUT UNTUK TULISAN DI WORD (SUB-BAB 4.1) ===
Lokasi file                      : C:\Users\HP\Documents\Tito\BAB IV\DATASET\data greenline 2024 sd juni 2026.xlsx
Total Baris Data Mentah (N_raw) : 13238 baris
Jumlah Sel Kosong (Null Check)   : 0 sel


In [7]:
# ------------------------------------------------------------------------------
# SEL 3: FEATURE ENGINEERING
# ------------------------------------------------------------------------------
df_feat = df_sorted.copy()

# A. Fitur Temporal
df_feat['day_of_week'] = df_feat['tanggal'].dt.dayofweek  # 0=Senin, 6=Minggu
df_feat['month'] = df_feat['tanggal'].dt.month
df_feat['is_weekend'] = df_feat['day_of_week'].apply(
    lambda x: 1 if x >= 5 else 0
)

# B. Label Encoding Nama Stasiun (0-19)
le_stasiun = LabelEncoder()
df_feat['stasiun_encoded'] = le_stasiun.fit_transform(df_feat['nama_stasiun'])

# C. Lag Features (H-1 dan H-7)
df_feat['lag_1'] = df_feat.groupby('nama_stasiun')['volume_penumpang'].shift(1)
df_feat['lag_7'] = df_feat.groupby('nama_stasiun')['volume_penumpang'].shift(7)

# D. Hapus baris NaN akibat pergeseran lag
df_clean = df_feat.dropna().reset_index(drop=True)

print("=== OUTPUT UNTUK TULISAN DI WORD (SUB-BAB 4.2.2) ===")
print(
    f"Total Baris Data Bersih Terintegrasi (N_clean): {len(df_clean)} baris\n"
)

# Tampilkan Daftar Mapping Label Encoding untuk Tabel 4.12
mapping_df = pd.DataFrame({
    'Kode Encoded': range(len(le_stasiun.classes_)),
    'Nama Stasiun': le_stasiun.classes_,
})
print("--- Mapping Label Encoding Stasiun (Tabel 4.12) ---")
print(mapping_df.to_string(index=False))

=== OUTPUT UNTUK TULISAN DI WORD (SUB-BAB 4.2.2) ===
Total Baris Data Bersih Terintegrasi (N_clean): 13098 baris

--- Mapping Label Encoding Stasiun (Tabel 4.12) ---
 Kode Encoded  Nama Stasiun
            0       CICAYUR
            1        CIKOYA
            2       CILEJIT
            3        CISAUK
            4       CITERAS
            5          DARU
            6        JATAKE
            7   JURANGMANGU
            8     KEBAYORAN
            9          MAJA
           10      PALMERAH
           11 PARUNGPANJANG
           12   PONDOKRANJI
           13 RANGKASBITUNG
           14    RAWA BUNTU
           15       SERPONG
           16      SUDIMARA
           17    TANAHABANG
           18         TENJO
           19     TIGARAKSA


In [8]:
# ------------------------------------------------------------------------------
# SEL 4: TIME-BASED SPLIT
# ------------------------------------------------------------------------------
cut_off_date = pd.to_datetime("2026-01-24")

# Bagi dataset secara kronologis
train_df = df_clean[df_clean["tanggal"] <= cut_off_date]
test_df = df_clean[df_clean["tanggal"] > cut_off_date]

feature_cols = [
    "stasiun_encoded",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_7",
]
target_col = "volume_penumpang"

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

print("=== OUTPUT UNTUK TULISAN DI WORD (SUB-BAB 4.2.3) ===")
print(
    f"Cut-off Date                 : {cut_off_date.strftime('%d %B %Y')}"
)
print(
    f"Jumlah Data Latih (X_train)  : {len(X_train)} baris ("
    f"{len(X_train)/len(df_clean)*100:.2f}%)"
)
print(
    f"Jumlah Data Uji   (X_test)   : {len(X_test)} baris ("
    f"{len(X_test)/len(df_clean)*100:.2f}%)"
)

=== OUTPUT UNTUK TULISAN DI WORD (SUB-BAB 4.2.3) ===
Cut-off Date                 : 24 January 2026
Jumlah Data Latih (X_train)  : 10488 baris (80.07%)
Jumlah Data Uji   (X_test)   : 2610 baris (19.93%)


In [9]:
# ------------------------------------------------------------------------------
# SEL 5: TRAIN RANDOM FOREST MODEL
# ------------------------------------------------------------------------------
rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=15, min_samples_split=5, random_state=42
)

# Training Model
rf_model.fit(X_train, y_train)

print(
    "✅ Model Random Forest berhasil dilatih dengan 200 pohon keputusan"
    " (n_estimators=200)."
)

✅ Model Random Forest berhasil dilatih dengan 200 pohon keputusan (n_estimators=200).


In [10]:
# ------------------------------------------------------------------------------
# SEL 6: SIMULASI 1 SAMPEL UNTUK PERHITUNGAN MANUAL (SUB-BAB 4.3.2)
# ------------------------------------------------------------------------------
# Ambil 1 sampel spesifik: Stasiun SERPONG pada 24 Jan 2026
sample_row = train_df[
    (train_df["nama_stasiun"] == "SERPONG")
    & (train_df["tanggal"] == "2026-01-24")
]

if len(sample_row) > 0:
  sample_x = sample_row[feature_cols]
  actual_y = sample_row[target_col].values[0]

  # Dapatkan nilai prediksi dari masing-masing 200 pohon keputusan
  tree_preds = [
      tree.predict(sample_x.values)[0] for tree in rf_model.estimators_
  ]

  sum_200_trees = sum(tree_preds)
  y_hat = sum_200_trees / 200
  abs_error = abs(actual_y - y_hat)

  print("=== ANGKA ASLI UNTUK DITULIS PADA SUB-BAB 4.3.2 WORD ===")
  print(f"Fitur Masukan (x)               : {sample_x.values[0].tolist()}")
  print(f"Volume Aktual (Y)               : {actual_y:,} orang")
  print(f"Hasil Pohon 1 (f1(x))           : {tree_preds[0]:,.2f} orang")
  print(f"Hasil Pohon 2 (f2(x))           : {tree_preds[1]:,.2f} orang")
  print(f"Hasil Pohon 3 (f3(x))           : {tree_preds[2]:,.2f} orang")
  print(f"Total Akumulasi 200 Pohon (sum) : {sum_200_trees:,.2f} orang")
  print(f"Hasil Prediksi Akhir Model(Y_hat): {y_hat:,.2f} orang")
  print(f"Absolute Error (|Y - Y_hat|)    : {abs_error:,.2f} orang")
else:
  print("⚠️ Sampel tanggal tersebut tidak ditemukan dalam dataset.")

=== ANGKA ASLI UNTUK DITULIS PADA SUB-BAB 4.3.2 WORD ===
Fitur Masukan (x)               : [15.0, 5.0, 1.0, 1.0, 11885.0, 10308.0]
Volume Aktual (Y)               : 10,366 orang
Hasil Pohon 1 (f1(x))           : 11,293.14 orang
Hasil Pohon 2 (f2(x))           : 10,294.60 orang
Hasil Pohon 3 (f3(x))           : 9,964.67 orang
Total Akumulasi 200 Pohon (sum) : 2,128,167.21 orang
Hasil Prediksi Akhir Model(Y_hat): 10,640.84 orang
Absolute Error (|Y - Y_hat|)    : 274.84 orang


In [11]:
# ------------------------------------------------------------------------------
# SEL 7: PEMBUKTIAN MATEMATIS EVALUASI (SUB-BAB 4.4 & TABEL 4.14)
# ------------------------------------------------------------------------------
# Prediksi Data Latih & Data Uji
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Hitung komponen rumus manual untuk Data Uji (n = len(X_test))
n_test = len(y_test)
sum_abs_error = np.sum(np.abs(y_test - y_pred_test))
ss_res = np.sum((y_test - y_pred_test) ** 2)
y_mean = np.mean(y_test)
ss_tot = np.sum((y_test - y_mean) ** 2)

# Metrik Evaluasi
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print("=== ANGKA PENJUMLAHAN ASLI UNTUK RUMUS SUB-BAB 4.4 WORD ===")
print(f"Total Sampel Data Uji (n)            : {n_test}")
print(f"Total Selisih Mutlak (sum|Y - Y_hat|): {sum_abs_error:,.2f}")
print(f"Jumlah Kuadrat Residual (SS_res)     : {ss_res:,.2f}")
print(f"Jumlah Kuadrat Total (SS_tot)        : {ss_tot:,.2f}")
print(f"Rata-rata Volume Aktual (Y_mean)     : {y_mean:,.2f}\n")

print("=== TAMPILAN TABEL 4.14 HASIL EVALUASI ===")
eval_table = pd.DataFrame({
    'Kategori Subset': ['Data Latih (Training Set)', 'Data Uji (Testing Set)'],
    'Jumlah Sampel (n)': [f'{len(X_train):,}', f'{len(X_test):,}'],
    'MAE (Penumpang)': [f'{mae_train:,.2f}', f'{mae_test:,.2f}'],
    'RMSE (Penumpang)': [f'{rmse_train:,.2f}', f'{rmse_test:,.2f}'],
    'R2 Score': [
        f'{r2_train:.4f} ({r2_train*100:.2f}%)',
        f'{r2_test:.4f} ({r2_test*100:.2f}%)',
    ],
})
print(eval_table.to_string(index=False))

=== ANGKA PENJUMLAHAN ASLI UNTUK RUMUS SUB-BAB 4.4 WORD ===
Total Sampel Data Uji (n)            : 2610
Total Selisih Mutlak (sum|Y - Y_hat|): 6,749,084.95
Jumlah Kuadrat Residual (SS_res)     : 67,773,698,661.65
Jumlah Kuadrat Total (SS_tot)        : 987,944,282,724.80
Rata-rata Volume Aktual (Y_mean)     : 19,860.60

=== TAMPILAN TABEL 4.14 HASIL EVALUASI ===
          Kategori Subset Jumlah Sampel (n) MAE (Penumpang) RMSE (Penumpang)        R2 Score
Data Latih (Training Set)            10,488        1,341.90         2,936.35 0.9775 (97.75%)
   Data Uji (Testing Set)             2,610        2,585.86         5,095.78 0.9314 (93.14%)


In [12]:
# ------------------------------------------------------------------------------
# SEL 7: PEMBUKTIAN MATEMATIS EVALUASI (SUB-BAB 4.4 & TABEL 4.14)
# ------------------------------------------------------------------------------
# Prediksi Data Latih & Data Uji
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Hitung komponen rumus manual untuk Data Uji (n = len(X_test))
n_test = len(y_test)
sum_abs_error = np.sum(np.abs(y_test - y_pred_test))
ss_res = np.sum((y_test - y_pred_test) ** 2)
y_mean = np.mean(y_test)
ss_tot = np.sum((y_test - y_mean) ** 2)

# Metrik Evaluasi
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print("=== ANGKA PENJUMLAHAN ASLI UNTUK RUMUS SUB-BAB 4.4 WORD ===")
print(f"Total Sampel Data Uji (n)            : {n_test}")
print(f"Total Selisih Mutlak (sum|Y - Y_hat|): {sum_abs_error:,.2f}")
print(f"Jumlah Kuadrat Residual (SS_res)     : {ss_res:,.2f}")
print(f"Jumlah Kuadrat Total (SS_tot)        : {ss_tot:,.2f}")
print(f"Rata-rata Volume Aktual (Y_mean)     : {y_mean:,.2f}\n")

print("=== TAMPILAN TABEL 4.14 HASIL EVALUASI ===")
eval_table = pd.DataFrame({
    'Kategori Subset': ['Data Latih (Training Set)', 'Data Uji (Testing Set)'],
    'Jumlah Sampel (n)': [f'{len(X_train):,}', f'{len(X_test):,}'],
    'MAE (Penumpang)': [f'{mae_train:,.2f}', f'{mae_test:,.2f}'],
    'RMSE (Penumpang)': [f'{rmse_train:,.2f}', f'{rmse_test:,.2f}'],
    'R2 Score': [
        f'{r2_train:.4f} ({r2_train*100:.2f}%)',
        f'{r2_test:.4f} ({r2_test*100:.2f}%)',
    ],
})
print(eval_table.to_string(index=False))

=== ANGKA PENJUMLAHAN ASLI UNTUK RUMUS SUB-BAB 4.4 WORD ===
Total Sampel Data Uji (n)            : 2610
Total Selisih Mutlak (sum|Y - Y_hat|): 6,749,084.95
Jumlah Kuadrat Residual (SS_res)     : 67,773,698,661.65
Jumlah Kuadrat Total (SS_tot)        : 987,944,282,724.80
Rata-rata Volume Aktual (Y_mean)     : 19,860.60

=== TAMPILAN TABEL 4.14 HASIL EVALUASI ===
          Kategori Subset Jumlah Sampel (n) MAE (Penumpang) RMSE (Penumpang)        R2 Score
Data Latih (Training Set)            10,488        1,341.90         2,936.35 0.9775 (97.75%)
   Data Uji (Testing Set)             2,610        2,585.86         5,095.78 0.9314 (93.14%)


In [13]:
# ------------------------------------------------------------------------------
# SEL 8: EXPORT MODEL ASET
# ------------------------------------------------------------------------------
joblib.dump(rf_model, 'model_rf_greenline.joblib')
joblib.dump(feature_cols, 'model_features.joblib')
joblib.dump(sorted(list(df_clean['nama_stasiun'].unique())), 'stations_list.joblib')

print(
    "💾 Berkas model_rf_greenline.joblib, model_features.joblib, dan"
    " stations_list.joblib berhasil disimpan!"
)

💾 Berkas model_rf_greenline.joblib, model_features.joblib, dan stations_list.joblib berhasil disimpan!


In [14]:
# ------------------------------------------------------------------------------
# IMPORT DATA EXCEL KE DATABASE SQLITE
# ------------------------------------------------------------------------------

from pathlib import Path
import sqlite3
import pandas as pd

# Notebook berada di folder:
# BAB IV/NOTEBOOK/

project_dir = Path.cwd().parent

excel_path = (
    project_dir
    / "DATASET"
    / "data greenline 2024 sd juni 2026.xlsx"
)

database_path = (
    project_dir
    / "DATASET"
    / "greenline.db"
)

if not excel_path.exists():
    raise FileNotFoundError(
        f"File Excel tidak ditemukan:\n{excel_path.resolve()}"
    )

# 1. Membaca Excel
df = pd.read_excel(excel_path)

# 2. Merapikan nama kolom
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

required_columns = {
    "tanggal",
    "nama_stasiun",
    "penumpang_berangkat_komuter",
    "penumpang_datang_komuter",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Kolom berikut tidak ditemukan: {sorted(missing_columns)}"
    )

# 3. Membersihkan nama stasiun
df["nama_stasiun"] = (
    df["nama_stasiun"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# 4. Membersihkan kolom jumlah penumpang
passenger_columns = [
    "penumpang_berangkat_komuter",
    "penumpang_datang_komuter",
]

for column in passenger_columns:
    df[column] = pd.to_numeric(
        df[column]
        .astype(str)
        .str.replace(r"[.,\s]", "", regex=True),
        errors="coerce",
    )

# 5. Membersihkan tanggal
df["tanggal"] = pd.to_datetime(
    df["tanggal"],
    dayfirst=True,
    errors="coerce",
)

# 6. Memeriksa data gagal dikonversi
invalid_data = df[
    df[
        [
            "tanggal",
            "nama_stasiun",
            "penumpang_berangkat_komuter",
            "penumpang_datang_komuter",
        ]
    ].isnull().any(axis=1)
]

if not invalid_data.empty:
    raise ValueError(
        f"Terdapat {len(invalid_data)} baris data yang tidak valid."
    )

# 7. Membuat volume penumpang
df["volume_penumpang"] = (
    df["penumpang_berangkat_komuter"]
    + df["penumpang_datang_komuter"]
)

# 8. Menyimpan tanggal dalam format SQLite
df["tanggal"] = df["tanggal"].dt.strftime("%Y-%m-%d")

# 9. Memilih kolom yang disimpan
df_database = df[
    [
        "tanggal",
        "nama_stasiun",
        "penumpang_berangkat_komuter",
        "penumpang_datang_komuter",
        "volume_penumpang",
    ]
].copy()

# Mencegah satu stasiun memiliki dua data pada tanggal yang sama
df_database = (
    df_database
    .drop_duplicates(
        subset=["nama_stasiun", "tanggal"],
        keep="last",
    )
    .sort_values(["nama_stasiun", "tanggal"])
    .reset_index(drop=True)
)

# 10. Menyimpan ke SQLite
with sqlite3.connect(database_path) as connection:
    df_database.to_sql(
        name="passenger_daily",
        con=connection,
        if_exists="replace",
        index=False,
    )

    connection.execute(
        """
        CREATE UNIQUE INDEX IF NOT EXISTS idx_station_date
        ON passenger_daily (nama_stasiun, tanggal)
        """
    )

    connection.commit()

print("Database berhasil dibuat.")
print(f"Lokasi database : {database_path.resolve()}")
print(f"Jumlah data      : {len(df_database):,} baris")
print(
    f"Periode data     : "
    f"{df_database['tanggal'].min()} sampai "
    f"{df_database['tanggal'].max()}"
)
print(
    f"Jumlah stasiun   : "
    f"{df_database['nama_stasiun'].nunique()}"
)

Database berhasil dibuat.
Lokasi database : C:\Users\HP\Documents\Tito\BAB IV\DATASET\greenline.db
Jumlah data      : 13,238 baris
Periode data     : 2024-01-01 sampai 2026-06-05
Jumlah stasiun   : 20
